# Chapter 14 — Reliability versus validity

**WHAT:** Compare noisy repeated indicators with a latent simulated target.

**WHY:** Absences missingness report does not empirically establish measurement validity or construct validity.

**HOW:** Seeded synthetic data, an explicit diagnostic, and a reproducible figure; metrics exported to JSON.

**Data category:** Seeded **synthetic teaching experiment** created inside the notebook. Supplied observational datasets are archived as optional context, not automatically used for this experiment.

**Execution:** From the repository root, use `python scripts/run_all.py`, or open this notebook in Jupyter and set the working directory to the repository root before Run All.

**Outputs:** `outputs/figures/chapter_14__reliability_versus_validity.png` and `outputs/metrics/chapter_14__reliability_versus_validity__metrics.json`.

**Interpretation:** Read the notebook output with the stated assumptions and scientific boundary; execution is not independent validation.


In [ ]:
CHAPTER=14
TITLE='Reliability versus validity'
BOUNDARY='Repeatability is distinct from construct validity; latent truth is simulation-only.'
OUTPUT_DIR=__import__('pathlib').Path('outputs')
FIGURE_DIR=OUTPUT_DIR/'figures'
METRIC_DIR=OUTPUT_DIR/'metrics'
SLUG='reliability_versus_validity'

import json
import matplotlib
matplotlib.use('Agg')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import linprog
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.base import clone
from pathlib import Path
SEED=20261024+CHAPTER
rng=np.random.default_rng(SEED)
N=1000
x=np.linspace(0,1,60)
fig, ax=plt.subplots(figsize=(7.3,4.4))
metric={}
if CHAPTER==2:
 y=rng.binomial(1,.08,5000);score=np.clip(.05+.65*y+rng.normal(0,.18,len(y)),0,1);ks=[50,100,250,500,900]
 vals=[]
 for k in ks:
  decision=np.zeros(len(y),int);decision[np.argsort(score)[-k:]]=1
  vals.append((12*np.sum((decision==0)&(y==1))+np.sum((decision==1)&(y==0)))/len(y))
 ax.plot(ks,vals,'o-');ax.set(xlabel='Daily review capacity',ylabel='Simulated cost per case');metric={'best_capacity':ks[int(np.argmin(vals))],'minimum_cost':round(float(min(vals)),4)}
elif CHAPTER==8:
 z=rng.normal(loc=.35,scale=1,size=280);B=[50,200,800,2500];ses=[]
 for b in B:ses.append(np.std(np.mean(rng.choice(z,size=(b,len(z)),replace=True),axis=1),ddof=1))
 ax.plot(B,ses,'o-',label='Bootstrap SE');ax.axhline(z.std(ddof=1)/np.sqrt(len(z)),ls='--',label='Analytic SE');ax.legend();ax.set(xlabel='Bootstrap draws',ylabel='Estimated SE');metric={'bootstrap_se_final':round(float(ses[-1]),4)}
elif CHAPTER==11:
 X=rng.normal(size=(1000,2));Y=2*X[:,0]+.6*X[:,1]+rng.normal(size=1000);m=LinearRegression().fit(X[:700],Y[:700]);ss=[0,.5,1,1.5,2];err=[]
 for s in ss:
  Xt=rng.normal(size=(500,2))+s;Yt=2*Xt[:,0]+.6*Xt[:,1]+.6*s*Xt[:,0]+rng.normal(size=500);err.append(mean_absolute_error(Yt,m.predict(Xt)))
 ax.plot(ss,err,'o-');ax.set(xlabel='Shift strength',ylabel='Test MAE');metric={'base_mae':round(err[0],3),'high_shift_mae':round(err[-1],3)}
elif CHAPTER==12:
 p=rng.uniform(.03,.97,9000);y=rng.binomial(1,p);bins=np.linspace(0,1,11);xx=[];yy=[]
 for j in range(10):
  mask=(p>=bins[j])&(p<bins[j+1]);xx.append(p[mask].mean());yy.append(y[mask].mean())
 ax.plot([0,1],[0,1],ls='--',color='gray');ax.plot(xx,yy,'o-');ax.set(xlabel='Predicted probability',ylabel='Observed event frequency');metric={'brier_score':round(brier_score_loss(y,p),4)}
elif CHAPTER==14:
 latent=rng.normal(size=1200);a=latent+rng.normal(0,.4,1200);b=latent+rng.normal(0,.4,1200);c=-latent+rng.normal(0,.4,1200)
 ax.scatter(a[::8],b[::8],s=9,alpha=.4,label='Repeat measures');ax.scatter(a[::8],c[::8],s=9,alpha=.4,label='Opposing construct');ax.legend();ax.set(xlabel='Indicator A',ylabel='Comparison measure');metric={'repeat_correlation':round(float(np.corrcoef(a,b)[0,1]),3),'opposing_correlation':round(float(np.corrcoef(a,c)[0,1]),3)}
elif CHAPTER==15:
 pop=np.r_[rng.normal(1,1,7000),rng.normal(6,2,3000)];truth=pop.mean();a=[];b=[]
 for _ in range(400):
  a.append(rng.choice(pop,200,replace=False).mean());b.append(.7*rng.choice(pop[:7000],100,replace=False).mean()+.3*rng.choice(pop[7000:],100,replace=False).mean())
 ax.hist(a,bins=25,alpha=.55,label='SRS');ax.hist(b,bins=25,alpha=.55,label='Weighted stratified');ax.axvline(truth,ls='--',color='black');ax.legend();ax.set(xlabel='Sample estimate',ylabel='Replications');metric={'srs_rmse':round(float(np.mean((np.array(a)-truth)**2)**.5),3),'stratified_rmse':round(float(np.mean((np.array(b)-truth)**2)**.5),3)}
elif CHAPTER==16:
 z=rng.normal(size=4000);t=rng.binomial(1,1/(1+np.exp(-.8*z)));y=1.5*t+1.8*z+rng.normal(size=len(z));miss=rng.binomial(1,1/(1+np.exp(-(.2+.8*z+.5*t)))).astype(bool);effects=[]
 for cutoff in [0,.2,.4,.6,.8]:
  mask=miss if cutoff==0 else (miss & (rng.uniform(size=len(z))>cutoff));effects.append((y[mask&t.astype(bool)].mean()-y[mask&~t.astype(bool)].mean(),LinearRegression().fit(np.c_[t[mask],z[mask]],y[mask]).coef_[0]))
 ax.plot(range(5),[v[0] for v in effects],'o-',label='Unadjusted');ax.plot(range(5),[v[1] for v in effects],'o-',label='Observed-Z adjustment');ax.axhline(1.5,ls='--',color='gray');ax.legend();ax.set(xlabel='Attrition scenario index',ylabel='Estimated effect');metric={'adjusted_first':round(float(effects[0][1]),3)}
elif CHAPTER==17:
 z=rng.normal(size=8000);pr=1/(1+np.exp(-(-.6+1.2*z)));included=rng.binomial(1,pr).astype(bool);vals=[z.mean(),z[included].mean(),np.average(z[included],weights=1/pr[included])];ax.bar(['Population','Selected','IPW selected'],vals);ax.set(ylabel='Mean outcome');metric={'population_mean':round(vals[0],3),'unweighted_selected_mean':round(vals[1],3),'ipw_mean':round(vals[2],3)}
elif CHAPTER==19:
 X=rng.normal(size=(1300,6))*np.array([1,2,5,10,20,50]);y=(X[:,0]/1+X[:,1]/2+.5*rng.normal(size=1300)>0).astype(int);ids=rng.permutation(1300);tr,te=ids[:900],ids[900:];scores=[]
 for c in [0.01,.1,1,10]:
  from sklearn.linear_model import LogisticRegression
  p=make_pipeline(StandardScaler(),LogisticRegression(C=c,max_iter=300)).fit(X[tr],y[tr]);scores.append(np.mean(p.predict(X[te])==y[te]))
 ax.plot([.01,.1,1,10],scores,'o-');ax.set_xscale('log');ax.set(xlabel='Regularisation C (train-only scaler)',ylabel='Held-out accuracy');metric={'max_heldout_accuracy':round(float(max(scores)),3)}
elif CHAPTER==20:
 lam=rng.gamma(1.8,3,size=5000);counts=rng.poisson(lam);values,cts=np.unique(counts,return_counts=True);ax.bar(values[values<26],cts[values<26]/len(counts),alpha=.7,label='Observed');xx=np.arange(26);ax.plot(xx,stats.poisson.pmf(xx,counts.mean()),'o-',label='Poisson fit');ax.legend();ax.set(xlabel='Count',ylabel='Frequency');metric={'variance_mean_ratio':round(float(counts.var(ddof=1)/counts.mean()),3),'observed_zero_fraction':round(float(np.mean(counts==0)),3)}
elif CHAPTER==22:
 n=35;truth=2.0;hits=[]
 for _ in range(2000):
  z=rng.normal(truth,1,n);lo=z.mean()-1.96*z.std(ddof=1)/np.sqrt(n);hi=z.mean()+1.96*z.std(ddof=1)/np.sqrt(n);hits.append(lo<=truth<=hi)
 ax.bar(['Nominal','Empirical'],[.95,np.mean(hits)]);ax.set(ylim=(0,1.02),ylabel='Interval coverage');metric={'empirical_coverage':round(float(np.mean(hits)),4),'replications':2000}
elif CHAPTER==23:
 p0=[];p1=[]
 for _ in range(1000):
  a=rng.normal(size=35);p0.append(stats.ttest_ind(a,rng.normal(size=35),equal_var=False).pvalue);p1.append(stats.ttest_ind(a,rng.normal(.5,1,35),equal_var=False).pvalue)
 ax.bar(['False rejection (null)','Power (effect=0.5)'],[np.mean(np.array(p0)<.05),np.mean(np.array(p1)<.05)]);ax.set(ylim=(0,1),ylabel='Proportion rejecting at 0.05');metric={'type_i_error':round(float(np.mean(np.array(p0)<.05)),3),'power':round(float(np.mean(np.array(p1)<.05)),3)}
elif CHAPTER==25:
 k=18;se=rng.uniform(.12,.33,k);true=rng.normal(.3,.32,k);est=true+rng.normal(size=k)*se;w=1/se**2;fixed=np.sum(w*est)/sum(w);Q=np.sum(w*(est-fixed)**2);C=sum(w)-sum(w*w)/sum(w);tau=max(0,(Q-(k-1))/C);wr=1/(se**2+tau);random=np.sum(wr*est)/sum(wr)
 ax.errorbar(np.arange(k),est,yerr=1.96*se,fmt='o',alpha=.6);ax.axhline(fixed,label='Fixed');ax.axhline(random,ls='--',label='Random');ax.legend();ax.set(xlabel='Synthetic study',ylabel='Effect and nominal 95% CI');metric={'fixed_effect':round(float(fixed),3),'random_effect':round(float(random),3),'tau_sq':round(float(tau),4)}
elif CHAPTER==26:
 rr=[];mm=[];levels=np.arange(0,.31,.05)
 for rate in levels:
  a=rng.normal(size=4000);n=int(len(a)*rate);a[:n]+=rng.normal(15,2,size=n);rr.append(a.mean());mm.append(np.median(a))
 ax.plot(levels,rr,'o-',label='Mean');ax.plot(levels,mm,'o-',label='Median');ax.legend();ax.set(xlabel='Contaminated fraction',ylabel='Location estimate');metric={'mean_at_max_contamination':round(float(rr[-1]),3),'median_at_max_contamination':round(float(mm[-1]),3)}
elif CHAPTER==28:
 from sklearn.model_selection import KFold
 n=4200;X=rng.normal(size=(n,3));e=1/(1+np.exp(-(.3+.7*X[:,0]-.5*X[:,1])));t=rng.binomial(1,e);truth=1.8;y=truth*t+X[:,0]+.6*X[:,1]+rng.normal(size=n)
 fold=KFold(n_splits=3,shuffle=True,random_state=SEED);phi=np.zeros(n)
 for tr,te in fold.split(X):
  pm=LogisticRegression(max_iter=400).fit(X[tr],t[tr]);eh=np.clip(pm.predict_proba(X[te])[:,1],.05,.95)
  m1=Ridge().fit(X[tr][t[tr]==1],y[tr][t[tr]==1]);m0=Ridge().fit(X[tr][t[tr]==0],y[tr][t[tr]==0]);v1=m1.predict(X[te]);v0=m0.predict(X[te]);phi[te]=v1-v0+t[te]*(y[te]-v1)/eh-(1-t[te])*(y[te]-v0)/(1-eh)
 naive=y[t==1].mean()-y[t==0].mean();ate=phi.mean();se=phi.std(ddof=1)/np.sqrt(n);ax.bar(['Unadjusted','Cross-fitted AIPW','Simulation truth'],[naive,ate,truth]);ax.errorbar([1],[ate],yerr=[1.96*se],fmt='none',color='black');ax.set(ylabel='Effect estimate');metric={'naive':round(float(naive),3),'aipw':round(float(ate),3),'aipw_se':round(float(se),4),'truth_for_simulation_evaluation_only':truth}
elif CHAPTER==30:
 clusters=120;rep=7;unit=rng.normal(size=clusters);data=np.repeat(unit,rep)+rng.normal(size=clusters*rep);naive=data.std(ddof=1)/np.sqrt(len(data));means=np.mean(data.reshape(clusters,rep),axis=1);clse=means.std(ddof=1)/np.sqrt(clusters);ax.bar(['Naive independent SE','Cluster-respecting SE'],[naive,clse]);ax.set(ylabel='Estimated SE of mean');metric={'naive_se':round(float(naive),4),'cluster_se':round(float(clse),4)}
elif CHAPTER==31:
 t=np.arange(450);y=15+2.2*np.sin(2*np.pi*t/12)+.015*t+rng.normal(0,.8,len(t));origins=range(180,420,12);na=[];ar=[]
 for o in origins:
  training=y[:o];lags=np.array([training[j-12:j] for j in range(12,len(training))]);target=training[12:];m=Ridge(alpha=4).fit(lags,target);na.append(abs(y[o]-y[o-12]));ar.append(abs(y[o]-m.predict(y[o-12:o].reshape(1,-1))[0]))
 ax.plot(list(origins),na,'o-',label='Seasonal naive |error|');ax.plot(list(origins),ar,'o-',label='Lagged ridge |error|');ax.legend();ax.set(xlabel='Origin (one-step forecast)',ylabel='Absolute error');metric={'seasonal_naive_mae':round(float(np.mean(na)),4),'ridge_mae':round(float(np.mean(ar)),4),'origins':len(na)}
elif CHAPTER==33:
 # 2-product finite resource allocation: 3 constraints and bounded decisions; exact LP for weighted objectives.
 A=np.array([[1,1],[2,1],[1,3]],float);b=np.array([90,130,160],float);weights=np.linspace(0,1,25);pts=[];residuals=[]
 for w in weights:
  c=np.array([4*w+1*(1-w),1*w+3*(1-w)]);res=linprog(-c,A_ub=A,b_ub=b,bounds=[(0,None),(0,None)],method='highs');assert res.success;pts.append(res.x);residuals.append(max(0,float(np.max(A@res.x-b))))
 pts=np.array(pts);ax.scatter(4*pts[:,0]+pts[:,1],pts[:,0]+3*pts[:,1],c=weights);ax.set(xlabel='Objective 1',ylabel='Objective 2');metric={'solver_successes':len(pts),'maximum_constraint_violation':round(float(max(residuals)),10),'unique_weighted_solutions':int(len(np.unique(np.round(pts,6),axis=0)))}
elif CHAPTER==36:
 corpus=['road rules permit buses at designated stations','statistical intervals describe uncertainty','a bank assesses capital adequacy','bus operating hours are published','statistical significance is not causal proof','public transport tickets vary by route','causal inference requires assumptions','retrieval grounded answers require evidence'];queries=['bus stops','statistical uncertainty','capital adequacy','bus timetable','causal significance','train tickets','causal assumptions','grounded evidence'];targets=[0,1,2,3,4,5,6,7];distractors=['bus causal proof','statistical bank stops','capital evidence bus','hours uncertainty','significance causal transport','tickets route travel','causal assumptions risk','evidence inference answers'];from sklearn.feature_extraction.text import TfidfVectorizer
 def acc(docs):
  v=TfidfVectorizer(ngram_range=(1,2));M=v.fit_transform(docs);Q=v.transform(queries);pred=(Q@M.T).toarray().argmax(1);return np.mean(pred==np.array(targets))
 scores=[acc(corpus),acc(corpus+distractors)];ax.bar(['No distractors','Added distractors'],scores);ax.set(ylim=(0,1),ylabel='Precision@1 (8 simulated queries)');metric={'no_distractor_p1':round(float(scores[0]),3),'with_distractors_p1':round(float(scores[1]),3)}
elif CHAPTER==39:
 a=np.array([2,4,10]);b=np.array([1,2,5]);ax.bar(np.arange(3)-0.2,a,.4,label='Untreated events per 100');ax.bar(np.arange(3)+0.2,b,.4,label='Treated events per 100');ax.set_xticks(range(3),['2% baseline','4% baseline','10% baseline']);ax.set(ylabel='Events per 100');ax.legend();metric={'relative_risk_reduction':'50% in all examples','absolute_risk_reductions_per_100':[1,2,5]}
elif CHAPTER==40:
 num=np.array([45,70,90,120]);den=np.array([100,150,200,250]);rate=num/den;ax.bar(['Q1','Q2','Q3','Q4'],rate);ax.set(ylim=(0,.6),ylabel='Reported services / eligible population');metric={'period_rates':np.round(rate,3).tolist(),'denominators':den.tolist()}
elif CHAPTER==41:
 days=np.arange(500);truth=10+np.sin(days/24);observed=truth+.003*days+rng.normal(0,.35,500);mask=rng.uniform(size=500)>.23;ax.plot(days,truth,label='Synthetic reference');ax.scatter(days[mask],observed[mask],s=3,label='Observed sensor');ax.legend();ax.set(xlabel='Time',ylabel='Reading');metric={'fraction_missing':round(float(np.mean(~mask)),3),'observed_bias':round(float(np.mean(observed[mask]-truth[mask])),3)}
elif CHAPTER==42:
 t=np.arange(300);demand=45+8*np.sin(2*np.pi*t/7)+rng.normal(0,4,len(t));origins=np.arange(84,292);pred=np.array([demand[i-7] for i in origins]);base=np.array([demand[:i].mean() for i in origins]);staff=np.ceil(pred/10)*10;cost=2*np.maximum(staff-demand[origins],0)+12*np.maximum(demand[origins]-staff,0);ax.plot(origins,cost,alpha=.7);ax.set(xlabel='Out-of-sample day',ylabel='Synthetic under/overstaffing cost');metric={'seasonal_mae':round(float(mean_absolute_error(demand[origins],pred)),3),'historical_mean_mae':round(float(mean_absolute_error(demand[origins],base)),3),'average_policy_cost':round(float(np.mean(cost)),3)}
elif CHAPTER==43:
 entities=np.repeat(['A','B','C'],12);period=np.tile(np.arange(12),3);left=pd.DataFrame({'entity':entities,'period':period,'numerator':rng.integers(20,50,36)});right=pd.DataFrame({'entity':entities,'period':period,'denominator':rng.integers(50,100,36)});right=right.drop(index=[4,12,20,32]);joined=left.merge(right,on=['entity','period'],how='left',validate='one_to_one',indicator=True);ax.bar(['Matched','Unmatched'],[(joined['_merge']=='both').sum(),(joined['_merge']!='both').sum()]);ax.set(ylabel='Institution-period records');metric={'matched_rows':int((joined['_merge']=='both').sum()),'unmatched_rows':int((joined['_merge']!='both').sum())}
elif CHAPTER==45:
 n=3000;X=rng.normal(size=(n,2));y=rng.binomial(1,1/(1+np.exp(-(X[:,0]+.5*X[:,1]))));m=LogisticRegression().fit(X[:2000],y[:2000]);rates=[];acc=[]
 for shift in [0,.5,1,1.5,2]:
  Xt=rng.normal(size=(1000,2))+shift;yt=rng.binomial(1,1/(1+np.exp(-(Xt[:,0]+.5*Xt[:,1]+.3*shift))));p=m.predict_proba(Xt)[:,1];conf=np.maximum(p,1-p);selected=conf>=.8;rates.append(np.mean(selected));acc.append(np.mean((p[selected]>=.5)==yt[selected]) if selected.any() else np.nan)
 ax.plot([0,.5,1,1.5,2],acc,'o-',label='Selected accuracy');ax.plot([0,.5,1,1.5,2],rates,'o-',label='Selection rate');ax.legend();ax.set(xlabel='Shift',ylabel='Rate');metric={'selected_accuracy_high_shift':round(float(acc[-1]),3),'selection_rate_high_shift':round(float(rates[-1]),3)}
elif CHAPTER==46:
 z=rng.standard_t(df=4,size=12000);levels=[.85,.9,.925,.95];shape=[];sizes=[]
 for q in levels:
  u=np.quantile(z,q);exc=z[z>u]-u;sizes.append(len(exc));params=stats.genpareto.fit(exc,floc=0);shape.append(params[0])
 ax.plot(levels,shape,'o-');ax.set(xlabel='Quantile threshold',ylabel='GPD fitted shape');metric={'shape_by_quantile':{str(v):round(float(s),3) for v,s in zip(levels,shape)},'smallest_tail_sample':min(sizes)}
elif CHAPTER==47:
 y=rng.binomial(1,.12,5000);s=np.clip(.1+.6*y+rng.normal(0,.25,5000),0,1);cost=[];thresholds=np.linspace(.1,.9,17)
 for th in thresholds:
  dec=s>=th;cost.append((12*np.sum((~dec)&(y==1))+np.sum(dec&(y==0)))/len(y))
 ax.plot(thresholds,cost,'o-');ax.set(xlabel='Action threshold',ylabel='Expected simulated loss per case');metric={'best_threshold':round(float(thresholds[np.argmin(cost)]),2),'minimum_loss':round(float(min(cost)),3)}
else: raise RuntimeError(CHAPTER)
ax.grid(alpha=.15);fig.tight_layout();FIGURE_DIR.mkdir(parents=True,exist_ok=True);METRIC_DIR.mkdir(parents=True,exist_ok=True);fig.savefig(FIGURE_DIR/f'chapter_{CHAPTER:02d}__{SLUG}.png',dpi=175);plt.close(fig)
print(json.dumps(metric,sort_keys=True))
(METRIC_DIR/f'chapter_{CHAPTER:02d}__{SLUG}__metrics.json').write_text(json.dumps({'chapter':CHAPTER,'seed':SEED,'dataset':'synthetic; constructed in notebook','experiment':TITLE,'results':metric,'limits':BOUNDARY},indent=2))


## Reading the results

**Interpretation guidance:** The figure illustrates **reliability versus validity** under a seeded synthetic data-generating process. Numeric metrics are saved separately as JSON so that they can be compared between executions.

**Scientific boundary:** Repeatability is distinct from construct validity; latent truth is simulation-only.

**Sensitivity task:** Change the seed or simulation assumptions, rerun from the repository root, and compare the resulting metric and figure. This is a teaching experiment, not a verified observational estimate or real-world causal claim.

**Repository data note:** Chapter-linked public datasets, when supplied, are listed in `datasets/DATASET_CATALOG.md`. They are optional materials unless explicitly loaded by code.
